# Crime Volume Trends

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from crime_snapshot import load_crime_snapshot
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
import plotly.graph_objects as go
warnings.filterwarnings('ignore')

df, metadata = load_crime_snapshot(
    PROJECT_ROOT / "data" / "processed" / "crime"
)

display(df.head())

CRIME_OUTPUT_DIR =  PROJECT_ROOT / "reports" / "crime"

TIME_COLUMN = "offense_date"
REPORT_TIME_COLUMN = "report_date_time"
EVENT_ID_COLUMN = "offense_id"
REPORT_ID_COLUMN = "report_number"
CATEGORY_COLUMN = "offense_category"

PLOTLY_TEMPLATE = "plotly_dark"
PLOT_BG = "#545455"
PAPER_BG = "#111111"

,report_number,report_date_time,offense_id,offense_date,nibrs_group_a_b,nibrs_crime_against_category,offense_sub_category,shooting_type_group,block_address,latitude,longitude,beat,precinct,sector,neighborhood,reporting_area,offense_category,nibrs_offense_code_description,nibrs_offense_code,census_block_2020
0,2025-216626,2025-07-30 21:29:01,65206207604,2025-07-30 19:40:00,a,person,aggravated assault,-,NW DOCK PL / RUSSELL AVE NW,47.665332,-122.379171,b1,north,b,ballard south,2582,violent crime,aggravated assault,13A,4701.3010
1,2025-216626,2025-07-30 21:29:01,65206263407,2025-07-30 19:40:00,a,property,motor vehicle theft,-,NW DOCK PL / RUSSELL AVE NW,47.665332,-122.379171,b1,north,b,ballard south,2582,property crime,motor vehicle theft,240,4701.3010
2,2025-216626,2025-07-30 21:29:01,65206225388,2025-07-30 19:40:00,a,property,"property offenses (includes stolen, destruction)",-,NW DOCK PL / RUSSELL AVE NW,47.665332,-122.379171,b1,north,b,ballard south,2582,all other,destruction/damage/vandalism of property,290,4701.3010
3,2025-216641,2025-07-30 20:39:42,65205847305,2025-07-30 19:50:00,a,property,larceny-theft,-,2XX BLOCK OF YALE AVE N,47.620244,-122.330420,d3,west,d,slu/cascade,3164,property crime,shoplifting,23C,7301.1005
4,2025-217214,2025-07-31 10:52:13,65214639215,2025-07-30 20:00:00,a,property,"property offenses (includes stolen, destruction)",-,79XX BLOCK OF 46TH AVE S,47.530306,-122.274475,s2,south,s,brighton/dunlap,402,all other,destruction/damage/vandalism of property,290,11801.2004


In [3]:
TIME_COLUMN = "offense_date"
REPORT_TIME_COLUMN = "report_date_time"
EVENT_ID_COLUMN = "offense_id"
REPORT_ID_COLUMN = "report_number"

required_columns = [
    TIME_COLUMN,
    REPORT_TIME_COLUMN,
    EVENT_ID_COLUMN,
    REPORT_ID_COLUMN,
    "offense_category",
    "offense_sub_category",
    "nibrs_crime_against_category",
    "nibrs_group_a_b",
    "nibrs_offense_code_description",
    "nibrs_offense_code",
    "precinct",
    "sector",
    "beat",
    "neighborhood",
    "reporting_area",
    "latitude",
    "longitude",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

missing_columns

[]

In [4]:
crime_df = df.copy()

crime_df[TIME_COLUMN] = pd.to_datetime(
    crime_df[TIME_COLUMN],
    errors="coerce",
)

crime_df[REPORT_TIME_COLUMN] = pd.to_datetime(
    crime_df[REPORT_TIME_COLUMN],
    errors="coerce",
)

crime_df[CATEGORY_COLUMN] = (
    crime_df[CATEGORY_COLUMN]
    .astype("string")
    .str.strip()
    .str.lower()
)

crime_df = crime_df.dropna(subset=[TIME_COLUMN, EVENT_ID_COLUMN])

crime_df.shape
display(crime_df[[TIME_COLUMN, EVENT_ID_COLUMN, REPORT_ID_COLUMN]].head())
date_min = crime_df[TIME_COLUMN].min()
date_max = crime_df[TIME_COLUMN].max()

print(f"Start date of calls: {date_min} \nEnd date: {date_max}")

record_count = len(crime_df)
unique_crime_events = crime_df[EVENT_ID_COLUMN].nunique()
unique_report_records = crime_df[REPORT_ID_COLUMN].nunique()

print(f"Number of records: {record_count} \nNumber of unique report records: {unique_report_records} \nNumber of unique crime events: {unique_crime_events}")

,offense_date,offense_id,report_number
0,2025-07-30 19:40:00,65206207604,2025-216626
1,2025-07-30 19:40:00,65206263407,2025-216626
2,2025-07-30 19:40:00,65206225388,2025-216626
3,2025-07-30 19:50:00,65205847305,2025-216641
4,2025-07-30 20:00:00,65214639215,2025-217214


Start date of calls: 2025-07-30 19:40:00 
End date: 2026-07-30 19:35:00
Number of records: 77307 
Number of unique report records: 65113 
Number of unique crime events: 77307


In [5]:
category_summary = (
    crime_df
    .groupby(CATEGORY_COLUMN, dropna=False)
    .agg(
        offense_count=(EVENT_ID_COLUMN, "size"),
        unique_offenses=(EVENT_ID_COLUMN, "nunique"),
        unique_reports=(REPORT_ID_COLUMN, "nunique"),
    )
    .reset_index()
    .sort_values("unique_offenses", ascending=False)
)

category_summary.head(30)

,offense_category,offense_count,unique_offenses,unique_reports
0,all other,36608,36608,29326
1,property crime,35618,35618,34918
2,violent crime,5081,5081,5056


In [6]:
selected_categories = (
    crime_df[CATEGORY_COLUMN]
    .dropna()
    .sort_values()
    .unique()
    .tolist()
)

selected_categories[:10], len(selected_categories)

(['all other', 'property crime', 'violent crime'], 3)

In [7]:
def build_crime_daily_volume(
    data: pd.DataFrame,
    selected_categories: list[str],
    time_column: str = TIME_COLUMN,
    event_id_column: str = EVENT_ID_COLUMN,
    category_column: str = CATEGORY_COLUMN,
) -> tuple[pd.DataFrame, dict]:
    working_df = data.copy()

    working_df[time_column] = pd.to_datetime(
        working_df[time_column],
        errors="coerce",
    )

    working_df[category_column] = (
        working_df[category_column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    working_df = working_df.dropna(
        subset=[
            time_column,
            event_id_column,
            category_column,
        ]
    )

    selected_categories = [
        str(category).strip().lower()
        for category in selected_categories
    ]

    filtered_df = working_df[
        working_df[category_column].isin(selected_categories)
    ].copy()

    if filtered_df.empty:
        raise ValueError("No records found for selected categories")

    filtered_df["date"] = filtered_df[time_column].dt.normalize()

    earliest_available_day = filtered_df["date"].min()
    latest_available_day = filtered_df["date"].max()

    # Match the calls-dashboard convention:
    # drop the earliest edge day because rolling snapshots may start mid-day.
    earliest_analysis_day = earliest_available_day + pd.Timedelta(days=1)

    past_year_start = latest_available_day - pd.Timedelta(days=364)

    plot_start_day = max(
        earliest_analysis_day,
        past_year_start,
    )

    plot_end_day = latest_available_day

    initial_view_start = max(
        plot_start_day,
        latest_available_day - pd.Timedelta(days=29),
    )

    plot_df = filtered_df[
        (filtered_df["date"] >= plot_start_day)
        & (filtered_df["date"] <= plot_end_day)
    ].copy()

    daily_volume = (
        plot_df
        .groupby("date")
        .agg(
            reported_offenses=(event_id_column, "nunique"),
            unique_reports=(REPORT_ID_COLUMN, "nunique"),
        )
        .reset_index()
    )

    full_date_range = pd.date_range(
        start=plot_start_day,
        end=plot_end_day,
        freq="D",
    )

    daily_volume = (
        daily_volume
        .set_index("date")
        .reindex(full_date_range)
        .rename_axis("date")
        .reset_index()
    )

    daily_volume["reported_offenses"] = (
        daily_volume["reported_offenses"]
        .fillna(0)
        .astype(int)
    )

    daily_volume["unique_reports"] = (
        daily_volume["unique_reports"]
        .fillna(0)
        .astype(int)
    )

    daily_volume["reported_offenses_7d_avg"] = (
        daily_volume["reported_offenses"]
        .rolling(window=7, min_periods=7)
        .mean()
    )

    date_context = {
        "earliest_available_day": earliest_available_day,
        "latest_available_day": latest_available_day,
        "earliest_analysis_day": earliest_analysis_day,
        "plot_start_day": plot_start_day,
        "plot_end_day": plot_end_day,
        "initial_view_start": initial_view_start,
        "selected_categories": selected_categories,
    }

    return daily_volume, date_context

In [8]:
daily_volume, date_context = build_crime_daily_volume(
    crime_df,
    selected_categories=selected_categories,
)

daily_volume.head(), daily_volume.tail(), date_context

(        date  reported_offenses  unique_reports  reported_offenses_7d_avg
 0 2025-07-31                218             184                       NaN
 1 2025-08-01                299             252                       NaN
 2 2025-08-02                243             200                       NaN
 3 2025-08-03                214             176                       NaN
 4 2025-08-04                214             183                       NaN,
           date  reported_offenses  unique_reports  reported_offenses_7d_avg
 360 2026-07-26                129              96                176.571429
 361 2026-07-27                113              93                160.714286
 362 2026-07-28                135             110                156.142857
 363 2026-07-29                115              84                143.285714
 364 2026-07-30                 49              38                124.428571,
 {'earliest_available_day': Timestamp('2025-07-30 00:00:00'),
  'latest_available_day'

In [15]:
selected_label = (
    "All selected categories"
    if len(selected_categories) > 1
    else selected_categories[0]
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=daily_volume["date"],
        y=daily_volume["reported_offenses"],
        name="Daily reported offenses",
        opacity=0.35,
        hovertemplate=(
            "<b>%{x|%b %d, %Y}</b><br>"
            "Reported offenses: %{y:,}<extra></extra>"
        ),
    )
)

fig.add_trace(
    go.Scatter(
        x=daily_volume["date"],
        y=daily_volume["reported_offenses_7d_avg"],
        mode="lines",
        name="7-day average",
        line=dict(width=3),
        hovertemplate=(
            "<b>%{x|%b %d, %Y}</b><br>"
            "7-day avg: %{y:.1f}<extra></extra>"
        ),
    )
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title=(
        "Daily Reported Crime Offenses"
        f"<br><sup>Category: {selected_label}</sup>"
    ),
    xaxis_title="Offense Date",
    yaxis_title="Reported Offenses",
    plot_bgcolor=PLOT_BG,
    paper_bgcolor=PAPER_BG,
    height=650,
    margin=dict(l=70, r=40, t=90, b=80),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
    ),
)

fig.update_xaxes(
    range=[
        date_context["initial_view_start"],
        date_context["plot_end_day"],
    ],
    rangeslider=dict(visible=True),
)

fig.show()

In [14]:
selected_categories = ['violent crime']

daily_volume, date_context = build_crime_daily_volume(
    crime_df,
    selected_categories=selected_categories,
)

daily_volume.head(), daily_volume.tail(), date_context

(        date  reported_offenses  unique_reports  reported_offenses_7d_avg
 0 2025-07-31                 24              24                       NaN
 1 2025-08-01                 29              29                       NaN
 2 2025-08-02                 18              18                       NaN
 3 2025-08-03                 21              21                       NaN
 4 2025-08-04                 12              12                       NaN,
           date  reported_offenses  unique_reports  reported_offenses_7d_avg
 360 2026-07-26                 16              15                 13.428571
 361 2026-07-27                 11              11                 12.857143
 362 2026-07-28                 16              16                 13.571429
 363 2026-07-29                  6               6                 13.142857
 364 2026-07-30                  3               3                 11.857143,
 {'earliest_available_day': Timestamp('2025-07-30 00:00:00'),
  'latest_available_day'

In [12]:
daily_by_category = (
    crime_df
    .dropna(subset=[TIME_COLUMN, CATEGORY_COLUMN])
    .assign(date=lambda x: x[TIME_COLUMN].dt.normalize())
    .groupby(["date", CATEGORY_COLUMN])
    .agg(reported_offenses=(EVENT_ID_COLUMN, "nunique"))
    .reset_index()
)

category_daily_stats = (
    daily_by_category
    .groupby(CATEGORY_COLUMN)
    .agg(
        active_days=("date", "nunique"),
        mean_daily_offenses=("reported_offenses", "mean"),
        median_daily_offenses=("reported_offenses", "median"),
        max_daily_offenses=("reported_offenses", "max"),
    )
    .reset_index()
    .sort_values("mean_daily_offenses", ascending=False)
)

category_daily_stats.head(30)

,offense_category,active_days,mean_daily_offenses,median_daily_offenses,max_daily_offenses
0,all other,366,100.021858,99.0,160
1,property crime,366,97.316940,98.0,148
2,violent crime,366,13.882514,13.0,29
